In [1]:
 def logImages(self, epoch_ndx, mode_str, dl):
#把模型设置为eval模式
        self.segmentation_model.eval()
#获取12个CT
        images = sorted(dl.dataset.series_list)[:12]
        for series_ndx, series_uid in enumerate(images):
            ct = getCt(series_uid)
#取出6个切片
            for slice_ndx in range(6):
                ct_ndx = slice_ndx * (ct.hu_a.shape[0] - 1) // 5
                sample_tup = dl.dataset.getitem_fullSlice(series_uid, ct_ndx)

                ct_t, label_t, series_uid, ct_ndx = sample_tup

                input_g = ct_t.to(self.device).unsqueeze(0)
                label_g = pos_g = label_t.to(self.device).unsqueeze(0)

                prediction_g = self.segmentation_model(input_g)[0]
                prediction_a = prediction_g.to('cpu').detach().numpy()[0] > 0.5
                label_a = label_g.cpu().numpy()[0][0] > 0.5

                ct_t[:-1,:,:] /= 2000
                ct_t[:-1,:,:] += 0.5

                ctSlice_a = ct_t[dl.dataset.contextSlices_count].numpy()

                image_a = np.zeros((512, 512, 3), dtype=np.float32)
                image_a[:,:,:] = ctSlice_a.reshape((512,512,1))
                image_a[:,:,0] += prediction_a & (1 - label_a) #把假阳性区域标记成红色
                image_a[:,:,0] += (1 - prediction_a) & label_a #假阴性标记为橙色
                image_a[:,:,1] += ((1 - prediction_a) & label_a) * 0.5 

                image_a[:,:,1] += prediction_a & label_a  #真阳性标记为绿色
                image_a *= 0.5
                image_a.clip(0, 1, image_a)

                writer = getattr(self, mode_str + '_writer')
                writer.add_image(
                    f'{mode_str}/{series_ndx}_prediction_{slice_ndx}',
                    image_a,
                    self.totalTrainingSamples_count,
                    dataformats='HWC',
                )

                if epoch_ndx == 1:
                    image_a = np.zeros((512, 512, 3), dtype=np.float32)
                    image_a[:,:,:] = ctSlice_a.reshape((512,512,1))
                    # image_a[:,:,0] += (1 - label_a) & lung_a # Red
                    image_a[:,:,1] += label_a  # Green
                    # image_a[:,:,2] += neg_a  # Blue

                    image_a *= 0.5
                    image_a[image_a < 0] = 0
                    image_a[image_a > 1] = 1
                    writer.add_image(
                        '{}/{}_label_{}'.format(
                            mode_str,
                            series_ndx,
                            slice_ndx,
                        ),
                        image_a,
                        self.totalTrainingSamples_count,
                        dataformats='HWC',
                    )

                writer.flush()

In [ ]:
 def saveModel(self, type_str, epoch_ndx, isBest=False):
#存储文件路径信息
        file_path = os.path.join(
            'data-unversioned',
            'part',
            'models',
            self.cli_args.tb_prefix,
            '{}_{}_{}.{}.state'.format(
                type_str,
                self.time_str,
                self.cli_args.comment,
                self.totalTrainingSamples_count,
            )
        )
#创建目录
        os.makedirs(os.path.dirname(file_path), mode=0o755, exist_ok=True)
#获取模型
        model = self.segmentation_model
        if isinstance(model, torch.nn.DataParallel):
            model = model.module
#需要存储的状态信息
        state = {
            'sys_argv': sys.argv,  #系统参数
            'time': str(datetime.datetime.now()), #时间信息
            'model_state': model.state_dict(), #模型状态
            'model_name': type(model).__name__, #模型名称
            'optimizer_state' : self.optimizer.state_dict(), #优化器状态
            'optimizer_name': type(self.optimizer).__name__, #优化器名称
            'epoch': epoch_ndx, #迭代周期
            'totalTrainingSamples_count': self.totalTrainingSamples_count, #训练样本数量
        }
#存储，通过存储模型，我们可以在下次接着训练
        torch.save(state, file_path)

        log.info("Saved model params to {}".format(file_path))
#这里做一个备份，如果这是效果最好的一版模型，就再存一次，记得多做这种操作，并且文件命名一定要好，具体为什么你自己考虑，说多了都是泪。
        if isBest:
            best_path = os.path.join(
                'data-unversioned', 'part', 'models',
                self.cli_args.tb_prefix,
                f'{type_str}_{self.time_str}_{self.cli_args.comment}.best.state')
            shutil.copyfile(file_path, best_path)

            log.info("Saved model params to {}".format(best_path))
#最后这个hash是用于校验文件的
        with open(file_path, 'rb') as f:
            log.info("SHA1: " + hashlib.sha1(f.read()).hexdigest())

In [ ]:
def main(self):
……
self.validation_cadence = 5
        for epoch_ndx in range(1, self.cli_args.epochs + 1):
        ……
            trnMetrics_t = self.doTraining(epoch_ndx, train_dl)
            self.logMetrics(epoch_ndx, 'trn', trnMetrics_t)
#记录第一个epoch或者每隔几个周期的时候记录图像信息
            if epoch_ndx == 1 or epoch_ndx % self.validation_cadence == 0:
                # if validation is wanted
                valMetrics_t = self.doValidation(epoch_ndx, val_dl)
                score = self.logMetrics(epoch_ndx, 'val', valMetrics_t)
                best_score = max(score, best_score)

                self.saveModel('seg', epoch_ndx, score == best_score)

                self.logImages(epoch_ndx, 'trn', train_dl)
                self.logImages(epoch_ndx, 'val', val_dl)